# Research Question : How did hospital emergency admissions change across major diagnosis categories before, during, and after the COVID-19 lockdown period? #

# Install Packages #

In [16]:
import sys
import subprocess
import importlib

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "plotly": "plotly",
    "kaleido": "kaleido"
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Imports #

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import textwrap
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

# Settings #

In [18]:
DATA_DIR = Path(".")
OUTPUT_DIR = DATA_DIR / "cw2_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SHEET_3CHAR = "Primary Diagnosis 3 Character"
SHEET_4CHAR = "Primary Diagnosis 4 Character"

YEARS = ["2018-19", "2019-20", "2020-21", "2021-22", "2022-23"]
BASELINE_YEAR = "2018-19"
LOCKDOWN_YEAR = "2020-21"
RECOVERY_YEAR = "2022-23"

METRIC = "emergency"   # change to "admissions" if needed

TOP_N_HEATMAP = 15
TOP_N_PARCOORDS = 8
TOP_PARENT_COUNT_TREEMAP = 6
TOP_LEAVES_PER_PARENT_TREEMAP = 3
MIN_BASELINE_VALUE = 100

LABEL_WRAP = 34
PARENT_LABEL_WRAP = 24
LEAF_LABEL_WRAP = 26

SHOW_HEATMAP_CELL_TEXT = False
SHOW_TREEMAP_VALUES = False

DROPDOWN_BG = "#dbe7f3"
DROPDOWN_BORDER = "#4c78a8"
DROPDOWN_TEXT = "#1f3552"

# File Discovery #

In [19]:
file_paths = sorted(DATA_DIR.glob("hosp-epis-stat-admi-diag*.xlsx"))

print("Working directory:", Path.cwd())
print("\nFiles found:")
for f in file_paths:
    print("-", f.name)

if not file_paths:
    raise FileNotFoundError("No Excel files found in the current notebook folder.")

Working directory: /Users/adityaraj/Desktop/ResearchMethods-CW2

Files found:
- hosp-epis-stat-admi-diag-2018-19-tab.xlsx
- hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx
- hosp-epis-stat-admi-diag-2020-21-tab.xlsx
- hosp-epis-stat-admi-diag-2021-22-tab.xlsx
- hosp-epis-stat-admi-diag-2022-23-tab_V2.xlsx


# Helper Function #

In [20]:
def clean_column_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    return col.strip("_")

def extract_year_label(filename):
    match = re.search(r"(\d{4}-\d{2})", filename)
    return match.group(1) if match else filename

def wrap_label(text, width=34):
    return "<br>".join(textwrap.wrap(str(text), width=width))

def short_label(text, max_len=44):
    text = str(text)
    return text if len(text) <= max_len else text[:max_len - 1] + "…"

def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def pick_existing_column(df, candidates, default_zero=False):
    for c in candidates:
        if c in df.columns:
            return c
    if default_zero:
        name = candidates[0]
        df[name] = 0
        return name
    raise KeyError(f"None of these columns were found: {candidates}")

def safe_write_image(fig, path, width=1800, height=1000, scale=2):
    try:
        fig.write_image(str(path), width=width, height=height, scale=scale)
        print("Saved:", path.name)
    except Exception as e:
        print(f"PNG export failed for {path.name}")
        print("Reason:", e)

def load_and_clean_excel(file_path, sheet_name):
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=10)

    keep_cols = [0, 1] + list(range(7, df.shape[1]))
    df = df.iloc[:, keep_cols].copy()

    df = df.rename(columns={
        df.columns[0]: "diagnosis_code",
        df.columns[1]: "diagnosis_description"
    })

    df.columns = [clean_column_name(c) for c in df.columns]

    rename_map = {
        "emergency_1": "zero_bed_day_cases_emergency",
        "elective": "zero_bed_day_cases_elective",
        "other": "zero_bed_day_cases_other",
        "age_90": "age_90_plus"
    }
    df = df.rename(columns=rename_map)

    df = df.dropna(how="all")
    df = df.dropna(subset=["diagnosis_code", "diagnosis_description"], how="all")

    df["diagnosis_code"] = df["diagnosis_code"].astype(str).str.strip()
    df["diagnosis_description"] = df["diagnosis_description"].astype(str).str.strip()

    bad_codes = df["diagnosis_code"].str.lower().isin(["", "nan", "total"])
    df = df[~bad_codes].copy()

    non_numeric = ["diagnosis_code", "diagnosis_description"]
    numeric_cols = [c for c in df.columns if c not in non_numeric]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df[numeric_cols] = df[numeric_cols].fillna(0)

    df["year"] = extract_year_label(file_path.name)

    return df

# Load Datasets #

In [21]:
print("\nLoading 3-character sheets...")
df_3char = pd.concat(
    [load_and_clean_excel(fp, SHEET_3CHAR) for fp in file_paths],
    ignore_index=True
)

print("Loading 4-character sheets...")
df_4char = pd.concat(
    [load_and_clean_excel(fp, SHEET_4CHAR) for fp in file_paths],
    ignore_index=True
)

for df in [df_3char, df_4char]:
    num_cols = df.select_dtypes(include=[np.number]).columns
    txt_cols = df.select_dtypes(include=["object"]).columns
    df[num_cols] = df[num_cols].fillna(0)
    df[txt_cols] = df[txt_cols].fillna("")

print("\n3-char shape:", df_3char.shape)
print("4-char shape:", df_4char.shape)
print("Remaining NaN values in 3-char:", df_3char.isna().sum().sum())
print("Remaining NaN values in 4-char:", df_4char.isna().sum().sum())


Loading 3-character sheets...
Loading 4-character sheets...

3-char shape: (8048, 46)
4-char shape: (40260, 46)
Remaining NaN values in 3-char: 0
Remaining NaN values in 4-char: 0


# Standardize Columns #

In [22]:
emergency_col_3 = pick_existing_column(df_3char, ["emergency"])
admissions_col_3 = pick_existing_column(df_3char, ["admissions"])
mean_age_col_3 = pick_existing_column(df_3char, ["mean_age"], default_zero=True)
mean_los_col_3 = pick_existing_column(df_3char, ["mean_length_of_stay"], default_zero=True)
mean_wait_col_3 = pick_existing_column(df_3char, ["mean_time_waited"], default_zero=True)

df_3char = df_3char.rename(columns={
    emergency_col_3: "emergency",
    admissions_col_3: "admissions",
    mean_age_col_3: "mean_age",
    mean_los_col_3: "mean_length_of_stay",
    mean_wait_col_3: "mean_time_waited"
})

emergency_col_4 = pick_existing_column(df_4char, ["emergency"])
admissions_col_4 = pick_existing_column(df_4char, ["admissions"])
mean_age_col_4 = pick_existing_column(df_4char, ["mean_age"], default_zero=True)
mean_los_col_4 = pick_existing_column(df_4char, ["mean_length_of_stay"], default_zero=True)
mean_wait_col_4 = pick_existing_column(df_4char, ["mean_time_waited"], default_zero=True)

df_4char = df_4char.rename(columns={
    emergency_col_4: "emergency",
    admissions_col_4: "admissions",
    mean_age_col_4: "mean_age",
    mean_los_col_4: "mean_length_of_stay",
    mean_wait_col_4: "mean_time_waited"
})

# Prepare 3-character Aggregated Data #

In [23]:
agg_3 = (
    df_3char
    .groupby(["diagnosis_code", "diagnosis_description", "year"], as_index=False)
    .agg(
        emergency=("emergency", "sum"),
        admissions=("admissions", "sum"),
        mean_age=("mean_age", "mean"),
        mean_length_of_stay=("mean_length_of_stay", "mean"),
        mean_time_waited=("mean_time_waited", "mean")
    )
)

agg_3["label"] = agg_3["diagnosis_code"] + " – " + agg_3["diagnosis_description"]

wide_3 = (
    agg_3.pivot_table(
        index=["diagnosis_code", "diagnosis_description", "label"],
        columns="year",
        values=METRIC,
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for y in YEARS:
    if y not in wide_3.columns:
        wide_3[y] = 0

wide_3["baseline"] = wide_3[BASELINE_YEAR]
wide_3["total_all_years"] = wide_3[YEARS].sum(axis=1)

for y in YEARS:
    if y == BASELINE_YEAR:
        wide_3[f"pct_change_{y}"] = 0.0
    else:
        wide_3[f"pct_change_{y}"] = np.where(
            wide_3[BASELINE_YEAR] > 0,
            100 * (wide_3[y] - wide_3[BASELINE_YEAR]) / wide_3[BASELINE_YEAR],
            np.nan
        )

wide_3["lockdown_pct_change"] = wide_3[f"pct_change_{LOCKDOWN_YEAR}"]
wide_3["recovery_pct_change"] = wide_3[f"pct_change_{RECOVERY_YEAR}"]

summary_3 = (
    agg_3.groupby(["diagnosis_code", "diagnosis_description", "label"], as_index=False)
    .agg(
        total_emergency=("emergency", "sum"),
        total_admissions=("admissions", "sum"),
        mean_age=("mean_age", "mean"),
        mean_length_of_stay=("mean_length_of_stay", "mean"),
        mean_time_waited=("mean_time_waited", "mean")
    )
)

summary_3 = summary_3.merge(
    wide_3[[
        "diagnosis_code", "diagnosis_description", "label",
        "baseline",
        "lockdown_pct_change", "recovery_pct_change",
        *YEARS,
        "total_all_years"
    ]],
    on=["diagnosis_code", "diagnosis_description", "label"],
    how="left"
)

summary_3["short_label"] = summary_3["label"].apply(short_label)

# Heatmap #

In [24]:
heat_df = wide_3[wide_3["baseline"] >= MIN_BASELINE_VALUE].copy()
heat_df = heat_df.sort_values("total_all_years", ascending=False).head(TOP_N_HEATMAP).copy()
heat_df = heat_df.sort_values("lockdown_pct_change", ascending=True)

raw_matrix = heat_df[YEARS].copy()
change_matrix = heat_df[[f"pct_change_{y}" for y in YEARS]].copy()
change_matrix.columns = YEARS
change_matrix = change_matrix.replace([np.inf, -np.inf], np.nan).fillna(0)

heat_labels = [wrap_label(lbl, LABEL_WRAP) for lbl in heat_df["label"]]
raw_matrix.index = heat_labels
change_matrix.index = heat_labels

metric_label = METRIC.capitalize()

heatmap_kwargs = dict(
    z=change_matrix.values,
    x=YEARS,
    y=change_matrix.index.tolist(),
    customdata=raw_matrix.values,
    colorscale="RdBu_r",
    zmid=0,
    colorbar=dict(title="% change<br>vs 2018-19"),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Year: %{x}<br>"
        + metric_label + ": %{customdata:,.0f}<br>"
        "Change vs 2018-19: %{z:.1f}%<extra></extra>"
    )
)

if SHOW_HEATMAP_CELL_TEXT:
    heatmap_kwargs["text"] = raw_matrix.values
    heatmap_kwargs["texttemplate"] = "%{text:,.0f}"

fig_heatmap = go.Figure(data=go.Heatmap(**heatmap_kwargs))
fig_heatmap.update_layout(
    title=dict(
        text=(
            f"Heatmap of Hospital {metric_label} Admissions Across Five Years"
            "<br><sup>Rows: 3-character diagnosis categories | Columns: years | "
            "Colour: % change vs 2018-19 | Hover: raw values</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),
    xaxis_title=None,
    yaxis_title="Diagnosis category",
    margin=dict(t=130, l=40, r=40, b=40),
    height=max(850, 38 * len(change_matrix.index)),
    font=dict(size=12)
)
fig_heatmap.update_xaxes(side="top", tickfont=dict(size=12))
fig_heatmap.update_yaxes(tickfont=dict(size=11))
fig_heatmap.show()

fig_heatmap.write_html(str(OUTPUT_DIR / "Heatmap.html"))
heat_df.to_csv(OUTPUT_DIR / "Heatmap_plot_data.csv", index=False)

safe_write_image(
    fig_heatmap,
    OUTPUT_DIR / "Heatmap.png",
    width=2200,
    height=max(1200, 48 * len(change_matrix.index)),
    scale=2
)

Saved: Heatmap.png


**Visualization using Heatmap : Emergency admissions for the most frequent 3-character diagnosis categories change across 2018–19 to 2022–23.**

# Prepare Treemap Hierarchy #

In [25]:
parent_lookup = (
    df_3char[["diagnosis_code", "diagnosis_description"]]
    .drop_duplicates(subset=["diagnosis_code"])
    .rename(columns={
        "diagnosis_code": "parent_code",
        "diagnosis_description": "parent_description"
    })
)

df_4char["parent_code"] = df_4char["diagnosis_code"].str.extract(r"^([A-Z][0-9][A-Z0-9])")

df_4char = df_4char.merge(parent_lookup, on="parent_code", how="left")
df_4char["parent_description"] = df_4char["parent_description"].fillna(df_4char["parent_code"])

df_4char["child_desc_norm"] = df_4char["diagnosis_description"].map(normalize_text)
df_4char["parent_desc_norm"] = df_4char["parent_description"].map(normalize_text)
df_4char["is_x_child"] = df_4char["diagnosis_code"].str.fullmatch(r"[A-Z][0-9][A-Z0-9]\.X", na=False)
df_4char["duplicate_parent_child"] = (
    df_4char["is_x_child"] &
    (df_4char["child_desc_norm"] == df_4char["parent_desc_norm"])
)
df_4char = df_4char[~df_4char["duplicate_parent_child"]].copy()
df_4char = df_4char[df_4char["parent_code"].notna()].copy()

agg_4 = (
    df_4char
    .groupby(
        ["parent_code", "parent_description", "diagnosis_code", "diagnosis_description", "year"],
        as_index=False
    )
    .agg(
        emergency=("emergency", "sum"),
        admissions=("admissions", "sum"),
        mean_age=("mean_age", "mean"),
        mean_length_of_stay=("mean_length_of_stay", "mean"),
        mean_time_waited=("mean_time_waited", "mean")
    )
)

wide_4 = (
    agg_4.pivot_table(
        index=["parent_code", "parent_description", "diagnosis_code", "diagnosis_description"],
        columns="year",
        values=METRIC,
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for y in YEARS:
    if y not in wide_4.columns:
        wide_4[y] = 0

wide_4["baseline"] = wide_4[BASELINE_YEAR]
wide_4["lockdown"] = wide_4[LOCKDOWN_YEAR]
wide_4["recovery"] = wide_4[RECOVERY_YEAR]
wide_4["abs_change_lockdown_vs_baseline"] = wide_4["lockdown"] - wide_4["baseline"]
wide_4["pct_change_lockdown_vs_baseline"] = np.where(
    wide_4["baseline"] > 0,
    100 * (wide_4["lockdown"] - wide_4["baseline"]) / wide_4["baseline"],
    np.nan
)

parent_sizes = (
    wide_4.groupby(["parent_code", "parent_description"], as_index=False)
    .agg(parent_baseline=("baseline", "sum"))
    .sort_values("parent_baseline", ascending=False)
)

top_parents = parent_sizes.head(TOP_PARENT_COUNT_TREEMAP)

treemap_df = wide_4.merge(
    top_parents[["parent_code", "parent_description"]],
    on=["parent_code", "parent_description"],
    how="inner"
)

treemap_df["rank_in_parent"] = (
    treemap_df.groupby(["parent_code", "parent_description"])["baseline"]
    .rank(method="first", ascending=False)
)

treemap_df = treemap_df[
    (treemap_df["rank_in_parent"] <= TOP_LEAVES_PER_PARENT_TREEMAP) &
    (treemap_df["baseline"] >= MIN_BASELINE_VALUE)
].copy()

treemap_df["parent_label"] = (
    treemap_df["parent_code"] + " – " +
    treemap_df["parent_description"].apply(lambda x: wrap_label(x, width=PARENT_LABEL_WRAP))
)
treemap_df["leaf_label"] = (
    treemap_df["diagnosis_code"] + " – " +
    treemap_df["diagnosis_description"].apply(lambda x: wrap_label(x, width=LEAF_LABEL_WRAP))
)

q_low = treemap_df["pct_change_lockdown_vs_baseline"].quantile(0.10)
q_high = treemap_df["pct_change_lockdown_vs_baseline"].quantile(0.90)
treemap_df["pct_change_color"] = treemap_df["pct_change_lockdown_vs_baseline"].clip(q_low, q_high)

# Treemap #

In [26]:
root_title = f"Hospital {metric_label} admissions"

hovertemplate_treemap = (
    "<b>%{customdata[2]} – %{customdata[3]}</b><br>"
    "Parent: %{customdata[0]} – %{customdata[1]}<br>"
    + BASELINE_YEAR + ": %{customdata[4]:,.0f}<br>"
    + LOCKDOWN_YEAR + ": %{customdata[5]:,.0f}<br>"
    + RECOVERY_YEAR + ": %{customdata[6]:,.0f}<br>"
    "Absolute change: %{customdata[7]:,.0f}<br>"
    "Percent change: %{customdata[8]:.1f}%<extra></extra>"
)

fig_treemap = px.treemap(
    treemap_df,
    path=[px.Constant(root_title), "parent_label", "leaf_label"],
    values="baseline",
    color="pct_change_color",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    custom_data=[
        "parent_code",
        "parent_description",
        "diagnosis_code",
        "diagnosis_description",
        "baseline",
        "lockdown",
        "recovery",
        "abs_change_lockdown_vs_baseline",
        "pct_change_lockdown_vs_baseline"
    ]
)

fig_treemap.update_traces(
    textinfo="label+value" if SHOW_TREEMAP_VALUES else "label",
    root_color="lightgrey",
    hovertemplate=hovertemplate_treemap
)

fig_treemap.update_layout(
    title=dict(
        text=(
            f"Treemap of Hospital {metric_label} Admissions"
            f"<br><sup>Size: {BASELINE_YEAR} | Colour: % change in {LOCKDOWN_YEAR} vs {BASELINE_YEAR} | "
            "Hierarchy: 3-character → 4-character diagnosis</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),
    margin=dict(t=110, l=10, r=10, b=10),
    coloraxis_colorbar_title="% change",
    uniformtext=dict(minsize=10, mode="hide")
)

fig_treemap.show()

fig_treemap.write_html(str(OUTPUT_DIR / "Treemap.html"))
treemap_df.to_csv(OUTPUT_DIR / "Treemap_plot_data.csv", index=False)

safe_write_image(
    fig_treemap,
    OUTPUT_DIR / "Treemap.png",
    width=2000,
    height=1200,
    scale=2
)

Saved: Treemap.png


**Visualization using Treemap : Diagnosis subcategories were most affected by the 2020–2021 lockdown.**

# Parallel Coordinates #

In [29]:
pc_df = summary_3[summary_3[BASELINE_YEAR] >= MIN_BASELINE_VALUE].copy()
pc_df = pc_df.sort_values("total_all_years", ascending=False).head(TOP_N_PARCOORDS).copy()
pc_df = pc_df.sort_values("lockdown_pct_change", ascending=True).reset_index(drop=True)

pc_df["short_label"] = pc_df["label"].apply(lambda x: short_label(x, 42))

parallel_dimensions = [
    ("2018-19", "2018-19"),
    ("2019-20", "2019-20"),
    ("2020-21", "2020-21"),
    ("2021-22", "2021-22"),
    ("2022-23", "2022-23"),
    ("Tot Adm", "total_admissions"),
    ("Mean Age", "mean_age"),
    ("Mean LOS", "mean_length_of_stay"),
    ("Mean Wait", "mean_time_waited")
]

dimension_ranges = {}
for axis_label, col_name in parallel_dimensions:
    col_min = float(pc_df[col_name].min())
    col_max = float(pc_df[col_name].max())
    if col_min == col_max:
        col_max = col_min + 1
    dimension_ranges[col_name] = [col_min, col_max]

cmin = float(pc_df["lockdown_pct_change"].min())
cmax = float(pc_df["lockdown_pct_change"].max())
if cmin == cmax:
    cmax = cmin + 1

def build_dimensions(df_subset):
    dims = []
    for axis_label, col_name in parallel_dimensions:
        dims.append(
            dict(
                label=axis_label,
                values=df_subset[col_name],
                range=dimension_ranges[col_name]
            )
        )
    return dims

fig_parcoords = go.Figure()

fig_parcoords.add_trace(
    go.Parcoords(
        visible=True,
        name="All categories",
        line=dict(
            color=pc_df["lockdown_pct_change"],
            colorscale="RdBu_r",
            cmin=cmin,
            cmax=cmax,
            showscale=True,
            colorbar=dict(
                title="% change<br>2020-21<br>vs 2018-19"
            )
        ),
        labelfont=dict(size=12),
        tickfont=dict(size=10),
        rangefont=dict(size=10),
        dimensions=build_dimensions(pc_df)
    )
)

for _, row in pc_df.iterrows():
    one_row = pd.DataFrame([row])

    fig_parcoords.add_trace(
        go.Parcoords(
            visible=False,
            name=row["short_label"],
            line=dict(
                color=[row["lockdown_pct_change"]],
                colorscale="RdBu_r",
                cmin=cmin,
                cmax=cmax,
                showscale=False
            ),
            labelfont=dict(size=12),
            tickfont=dict(size=10),
            rangefont=dict(size=10),
            dimensions=build_dimensions(one_row)
        )
    )

buttons = []

buttons.append(
    dict(
        label="All categories",
        method="update",
        args=[
            {"visible": [True] + [False] * len(pc_df)},
            {
                "title": {
                    "text": (
                        "Parallel Coordinates Plot of Diagnosis Profiles"
                        "<br><sup>All selected categories | Brush on any axis to filter interactively</sup>"
                    ),
                    "x": 0.5,
                    "xanchor": "center",
                    "font": {"size": 20}
                }
            }
        ]
    )
)

for i, label in enumerate(pc_df["short_label"], start=1):
    visible = [False] * (len(pc_df) + 1)
    visible[i] = True

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {"visible": visible},
                {
                    "title": {
                        "text": (
                            f"Parallel Coordinates Plot: {label}"
                            "<br><sup>Single diagnosis category | All 5 years + key hospital attributes</sup>"
                        ),
                        "x": 0.5,
                        "xanchor": "center",
                        "font": {"size": 20}
                    }
                }
            ]
        )
    )

fig_parcoords.update_layout(
    title=dict(
        text=(
            "Parallel Coordinates Plot of Diagnosis Profiles"
            "<br><sup>All selected categories | Brush on any axis to filter interactively</sup>"
        ),
        x=0.56,
        xanchor="center",
        y=0.96,
        font=dict(size=20)
    ),
    margin=dict(t=230, l=70, r=70, b=40),
    height=900,
    font=dict(size=12),
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            active=0,
            x=-0.02,
            xanchor="left",
            y=1.28,
            yanchor="top",
            bgcolor=DROPDOWN_BG,
            bordercolor=DROPDOWN_BORDER,
            borderwidth=2,
            font=dict(
                size=13,
                color=DROPDOWN_TEXT
            )
        )
    ],
    annotations=[
        dict(
            text="Select a diagnosis category:",
            x=-0.02,
            xanchor="left",
            y=1.35,
            yanchor="top",
            showarrow=False,
            font=dict(size=13, color=DROPDOWN_TEXT)
        )
    ]
)

fig_parcoords.show()

fig_parcoords.write_html(str(OUTPUT_DIR / "Parallel_coordinates.html"))
pc_df.to_csv(OUTPUT_DIR / "Parallel_coordinates_plot_data.csv", index=False)
pc_df[["short_label", "diagnosis_code", "diagnosis_description"]].to_csv(
    OUTPUT_DIR / "Parallel_coordinates_category_list.csv",
    index=False
)

fig_parcoords_static = go.Figure(
    data=go.Parcoords(
        line=dict(
            color=pc_df["lockdown_pct_change"],
            colorscale="RdBu_r",
            cmin=cmin,
            cmax=cmax,
            showscale=True,
            colorbar=dict(title="% change<br>2020-21<br>vs 2018-19")
        ),
        labelfont=dict(size=13),
        tickfont=dict(size=11),
        rangefont=dict(size=11),
        dimensions=build_dimensions(pc_df)
    )
)

fig_parcoords_static.update_layout(
    title=dict(
        text=(
            "Parallel Coordinates Plot of Diagnosis Profiles"
            "<br><sup>Selected categories | All 5 years + key hospital attributes</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),
    margin=dict(t=120, l=60, r=60, b=40),
    height=850,
    font=dict(size=12)
)

fig_parcoords_static.show()

safe_write_image(
    fig_parcoords_static,
    OUTPUT_DIR / "Parallel_coordinates.png",
    width=2800,
    height=1200,
    scale=2
)

Saved: Parallel_coordinates.png


**Visualization using Parallel Coordinates : Diagnosis categories compare across all five years and key hospital attributes when viewed individually and collectively.**

In [30]:
print("\n FILES CREATED \n")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f.name)


 FILES CREATED 

Heatmap.html
Heatmap.png
Heatmap_plot_data.csv
Parallel_coordinates.html
Parallel_coordinates.png
Parallel_coordinates_category_list.csv
Parallel_coordinates_plot_data.csv
Treemap.html
Treemap.png
Treemap_plot_data.csv
